### Trabalho Fase 2 do Curso de Pós-Graduação FIAP IA para Devs
#### Parte 9 - Tuning de Random Forest com Algoritmos Genéticos

Fonte de dados escolhida: DATASUS/SISCAN  
Tipo de dados de origem nesta etapa: Parquet

Arquivos utilizados:

```text
bases/treino/x_encoded.parquet
bases/treino/y.parquet
bases/teste/x_encoded.parquet
bases/teste/y.parquet
```

---

## Objetivo da Parte 9

O objetivo desta etapa é otimizar os hiperparâmetros do modelo Random Forest (`RandomForestClassifier`) usando um Algoritmo Genético implementado diretamente no notebook. A estrutura segue as Partes 7 e 8 como one-shot: mesma leitura de dados, mesma estratégia de amostragem, mesmos operadores genéticos e mesma função de fitness voltada ao desbalanceamento da target.

O cromossomo de cada indivíduo representa uma configuração da Random Forest:

- `n_estimators`;
- `max_depth`;
- `min_samples_leaf`;
- `class_weight`.

Breve descrição dos hiperparâmetros otimizados:

- `n_estimators`: define a quantidade de árvores na floresta. Mais árvores tendem a estabilizar as previsões e reduzir variância, mas aumentam o tempo de treino. O objetivo do AG é encontrar uma quantidade suficiente de árvores sem custo computacional desnecessário.
- `max_depth`: controla a profundidade máxima de cada árvore. Profundidades menores limitam a complexidade e podem reduzir overfitting; profundidades maiores permitem capturar interações mais específicas entre as variáveis. O objetivo é equilibrar capacidade preditiva e generalização.
- `min_samples_leaf`: define o número mínimo de amostras em cada folha. Valores maiores tornam as árvores mais conservadoras e menos sensíveis a ruído; valores menores permitem divisões mais específicas. O objetivo é controlar overfitting e melhorar a estabilidade da classe positiva.
- `class_weight`: define pesos diferenciados para as classes. As opções `balanced` e `balanced_subsample` aumentam a importância da classe minoritária, ajudando a Random Forest a lidar com o desbalanceamento da target. O objetivo é melhorar recall e F1 da classe positiva.

A coluna de idade mantida neste notebook é `CO_IDADE_PACIENTE_NUM`, alinhada ao melhor baseline de Random Forest da Parte 5. Como Random Forest não depende de distância entre observações, este notebook não aplica padronização nem normalização das features.

---

## Índice / Sumário da Parte 9

**Item 1 - Leitura e preparação dos dados para Random Forest**

- Leitura das bases encoded geradas na Parte 2.
- Definição de constantes, espaços de busca e métricas usadas na avaliação.
- Seleção das colunas usadas pela Random Forest.

**Item 2 - Amostragem estratificada**

- Criação de amostras estratificadas de treino e teste.
- Preservação aproximada da proporção da classe positiva.

**Item 3 - Execução da Random Forest com validação cruzada**

- Treinamento do `RandomForestClassifier` em 5 folds estratificados.
- Retorno das métricas médias dos folds.

**Item 4 - Função de fitness**

- Cálculo da fitness com a combinação `0.7 * F1 + 0.3 * recall`.

**Item 5 - Representação genética e população inicial**

- Definição do cromossomo, solução base, variação inicial e criação dos indivíduos.

**Item 6 - Operadores genéticos**

- Avaliação da população.
- Seleção por torneio.
- Crossover uniforme.
- Mutação por gene.
- Elitismo e geração da nova população.

**Item 7 - Experimentos com Algoritmo Genético**

- Execução de três configurações de população, mutação e número de gerações.
- Registro do melhor indivíduo por geração.

**Item 8 - Avaliação final da Random Forest otimizada**

- Escolha do melhor indivíduo entre os três experimentos.
- Treinamento final sem validação cruzada.
- Avaliação na amostra estratificada de teste.

**Item 9 - Comparação com a Parte 5 e conclusão**

- Comparação entre o baseline de Random Forest da Parte 5 e a Random Forest otimizada por Algoritmo Genético.
- Síntese dos ganhos, perdas e trade-offs observados nas métricas da classe positiva.


#### Item 1 - Leitura e preparação dos dados para Random Forest

A Parte 9 usa diretamente `x_encoded.parquet`, criado na Parte 2. Como a validação das bases já foi realizada na etapa de preparação, este bloco concentra os imports, a configuração das métricas, a leitura das bases e a definição dos espaços de busca do Algoritmo Genético.

Também é definida a estrutura `ClassificationMetrics`, usada para padronizar as métricas calculadas ao longo do notebook.


In [1]:
from pathlib import Path

import random
import time
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
try:
    from IPython.display import display, HTML
except ImportError:
    def display(value):
        print(value)

    class HTML(str):
        pass
import numpy as np

from dataclasses import dataclass

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

random.seed(42)
np.random.seed(42)

display(HTML("""
<style>
.container {
    width: 100% !important;
    max-width: 100% !important;
}
.jp-Cell-outputArea,
.jp-OutputArea-output,
.output_area {
    width: 100% !important;
    max-width: 100% !important;
}
.dataframe {
    width: 100% !important;
}
.dataframe td,
.dataframe th {
    white-space: normal !important;
    word-break: break-word !important;
}
</style>
"""))

@dataclass
class ClassificationMetrics:
    accuracy: float
    recall: float
    f1: float
    precision: float
    roc_auc: float
    balanced_accuracy: float


def calculate_metrics(y_true, y_pred, y_score=None) -> ClassificationMetrics:
    roc_auc_input = y_score if y_score is not None else y_pred
    return ClassificationMetrics(
        accuracy=accuracy_score(y_true, y_pred),
        recall=recall_score(y_true, y_pred, zero_division=0),
        f1=f1_score(y_true, y_pred, zero_division=0),
        precision=precision_score(y_true, y_pred, zero_division=0),
        roc_auc=roc_auc_score(y_true, roc_auc_input),
        balanced_accuracy=balanced_accuracy_score(y_true, y_pred),
    )


N_ESTIMATORS_SPACE = [100, 200, 300]
MAX_DEPTH_SPACE = [8, 12, 16, None]
MIN_SAMPLES_LEAF_SPACE = [1, 5, 10, 20]
CLASS_WEIGHT_SPACE = [None, "balanced", "balanced_subsample"]

RANDOM_STATE = 42
TARGET_COLUMN = 'TARGET_CANCER_MAMA_PROVAVEL'
DATA_DIR = Path('bases')

N_TRAIN_RF = 12000
N_VALIDATION_RF = 4000
N_TEST_RF = 8000

input_files = {
    'x_train_encoded': DATA_DIR / 'treino' / 'x_encoded.parquet',
    'y_train': DATA_DIR / 'treino' / 'y.parquet',
    'x_test_encoded': DATA_DIR / 'teste' / 'x_encoded.parquet',
    'y_test': DATA_DIR / 'teste' / 'y.parquet',
}

X_train_encoded = pd.read_parquet(input_files['x_train_encoded'])
y_train = pd.read_parquet(input_files['y_train'])[TARGET_COLUMN]
X_test_encoded = pd.read_parquet(input_files['x_test_encoded'])
y_test = pd.read_parquet(input_files['y_test'])[TARGET_COLUMN]

X_train_encoded = X_train_encoded.drop(columns=['CO_IDADE_PACIENTE_CAP'])
X_test_encoded = X_test_encoded.drop(columns=['CO_IDADE_PACIENTE_CAP'])

age_columns = ['CO_IDADE_PACIENTE_NUM']

context_numeric_columns = [
    column for column in ['CO_TEMPO_MAMO_ANTERIOR_NUM']
    if column in X_train_encoded.columns
]
encoded_categorical_columns = [
    column for column in X_train_encoded.columns
    if column not in age_columns + context_numeric_columns
]

rf_input_summary = {
    'x_train_encoded_shape': X_train_encoded.shape,
    'x_test_encoded_shape': X_test_encoded.shape,
    'selected_age_columns': age_columns,
    'context_numeric_column_count': len(context_numeric_columns),
    'context_numeric_columns': context_numeric_columns,
    'encoded_categorical_column_count': len(encoded_categorical_columns),
}
rf_input_summary


{'x_train_encoded_shape': (213978, 24),
 'x_test_encoded_shape': (53495, 24),
 'selected_age_columns': ['CO_IDADE_PACIENTE_NUM'],
 'context_numeric_column_count': 1,
 'context_numeric_columns': ['CO_TEMPO_MAMO_ANTERIOR_NUM'],
 'encoded_categorical_column_count': 22}

#### Item 2 - Amostragem estratificada

A amostragem estratificada preserva aproximadamente a proporção da target, que é fortemente desbalanceada. A amostra de treino é usada pelo Algoritmo Genético com validação cruzada; a amostra de teste fica reservada para a avaliação final do melhor indivíduo. Para manter a comparação com a Parte 5, a amostra final de teste usa `N_TEST_RF = 8000` registros.


In [2]:
def limit_stratified_sample(X, y, sample_size, random_state=RANDOM_STATE):
    if sample_size is None or sample_size >= len(X):
        return X.copy(), y.copy()

    _, X_sample, _, y_sample = train_test_split(
        X,
        y,
        test_size=sample_size,
        random_state=random_state,
        stratify=y,
    )
    return X_sample.reset_index(drop=True), y_sample.reset_index(drop=True)


X_train_sample, y_train_sample = limit_stratified_sample(
    X_train_encoded,
    y_train,
    N_TRAIN_RF + N_VALIDATION_RF,
)

X_test_sample, y_test_sample = limit_stratified_sample(
    X_test_encoded,
    y_test,
    N_TEST_RF,
)

rf_samples_summary = pd.DataFrame([
    {'base': 'treino_rf', 'linhas': len(X_train_sample), 'classe_positiva': int(y_train_sample.sum()), 'percentual_positivo': round(y_train_sample.mean() * 100, 2)},
    {'base': 'teste_rf', 'linhas': len(X_test_sample), 'classe_positiva': int(y_test_sample.sum()), 'percentual_positivo': round(y_test_sample.mean() * 100, 2)},
])
rf_samples_summary


,base,linhas,classe_positiva,percentual_positivo
0,treino_rf,16000,679,4.24
1,teste_rf,8000,339,4.24


#### Item 3 - Execução da Random Forest com validação cruzada

O bloco abaixo reúne a avaliação do `RandomForestClassifier`. Para cada indivíduo, o modelo é treinado em 5 folds estratificados, e as métricas finais retornadas são as médias dos folds.

Essa função é usada durante o Algoritmo Genético para avaliar cada combinação de hiperparâmetros sem usar o conjunto de teste.


In [3]:
def execute_random_forest(
    x,
    y,
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=10,
    class_weight="balanced_subsample",
    show=False,
    short_show=False,
):
    n_splits = 5
    skf = StratifiedKFold(n_splits, shuffle=True, random_state=RANDOM_STATE)

    metrics = ClassificationMetrics(
        accuracy=0.0,
        recall=0.0,
        f1=0.0,
        precision=0.0,
        roc_auc=0.0,
        balanced_accuracy=0.0,
    )

    for train_idx, val_idx in skf.split(x, y):
        X_tr, X_val = x.iloc[train_idx], x.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        classifier = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            class_weight=class_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

        classifier.fit(X_tr, y_tr)
        y_pred = classifier.predict(X_val)
        y_score = classifier.predict_proba(X_val)[:, 1]

        current_execution = calculate_metrics(y_val, y_pred, y_score)

        metrics.accuracy += current_execution.accuracy
        metrics.recall += current_execution.recall
        metrics.f1 += current_execution.f1
        metrics.precision += current_execution.precision
        metrics.roc_auc += current_execution.roc_auc
        metrics.balanced_accuracy += current_execution.balanced_accuracy

    metrics.accuracy /= n_splits
    metrics.recall /= n_splits
    metrics.f1 /= n_splits
    metrics.precision /= n_splits
    metrics.roc_auc /= n_splits
    metrics.balanced_accuracy /= n_splits

    if show:
        if short_show:
            print(f'recall: {metrics.recall} f1: {metrics.f1}')
        else:
            print(f'########### RandomForestClassifier ##############')
            print(f'n_estimators: {n_estimators}')
            print(f'max_depth: {max_depth}')
            print(f'min_samples_leaf: {min_samples_leaf}')
            print(f'class_weight: {class_weight}')
            print(f'accuracy: {metrics.accuracy}')
            print(f'recall: {metrics.recall}')
            print(f'f1: {metrics.f1}')
            print(f'precision: {metrics.precision}')
            print(f'roc_auc: {metrics.roc_auc}')
            print(f'balanced_accuracy: {metrics.balanced_accuracy}')
            print('#################################################')

    return metrics


#### Item 4 - Função de fitness

A fitness prioriza a classe positiva, que é minoritária. A métrica principal combina F1 e recall:

```text
fitness = 0.7 * F1 + 0.3 * recall
```

O F1 equilibra precision e recall, enquanto o recall aumenta a importância de recuperar casos positivos.


In [4]:
def fitness(individual, show=False, short_show=False) -> float:
    rf = execute_random_forest(
        x=X_train_sample,
        y=y_train_sample,
        n_estimators=individual["n_estimators"],
        max_depth=individual["max_depth"],
        min_samples_leaf=individual["min_samples_leaf"],
        class_weight=individual["class_weight"],
        show=show,
        short_show=short_show,
    )

    return calculate_rf_execution_fitness(rf)


def calculate_rf_execution_fitness(rf_execution):
    return 0.7 * rf_execution.f1 + 0.3 * rf_execution.recall


#### Item 5 - Representação genética e população inicial

Cada indivíduo representa um conjunto de hiperparâmetros da Random Forest. A população inicial é criada a partir de uma solução base com variações aleatórias, estratégia usada como hot start para iniciar a busca em uma região plausível do espaço de hiperparâmetros.


In [5]:
def create_random_individuals(individual_count):
    return [{
        "n_estimators": random.choice(N_ESTIMATORS_SPACE),
        "max_depth": random.choice(MAX_DEPTH_SPACE),
        "min_samples_leaf": random.choice(MIN_SAMPLES_LEAF_SPACE),
        "class_weight": random.choice(CLASS_WEIGHT_SPACE),
    } for _ in range(individual_count)]


BASE_RF_SOLUTION = {
    "n_estimators": 200,
    "max_depth": None,
    "min_samples_leaf": 10,
    "class_weight": "balanced_subsample",
}

VARIATION_RATE = 0.3


def create_rf_variation(base_solution, variation_rate):
    individual = base_solution.copy()

    if random.random() < variation_rate:
        individual["n_estimators"] = random.choice(N_ESTIMATORS_SPACE)

    if random.random() < variation_rate:
        individual["max_depth"] = random.choice(MAX_DEPTH_SPACE)

    if random.random() < variation_rate:
        individual["min_samples_leaf"] = random.choice(MIN_SAMPLES_LEAF_SPACE)

    if random.random() < variation_rate:
        individual["class_weight"] = random.choice(CLASS_WEIGHT_SPACE)

    return individual


def create_hot_start_individuals(individual_count):
    return [create_rf_variation(BASE_RF_SOLUTION, VARIATION_RATE) for _ in range(individual_count)]


#### Item 6.1 - Avaliação da população

Cada indivíduo recebe uma nota de fitness calculada pela validação cruzada. Em seguida, a população é ordenada da maior para a menor fitness, permitindo selecionar os melhores candidatos para reprodução e elitismo.


In [6]:
def evaluate_population(individuals):   
    for individual in individuals:
        individual['evaluation'] = fitness(individual)

    sorted_population = sorted(
        individuals,
        key=lambda item: item["evaluation"],
        reverse=True
    )
    
    return sorted_population           


#### Item 6.2 - Seleção por torneio

A seleção por torneio escolhe alguns candidatos aleatoriamente e retorna o indivíduo com maior fitness entre eles. Esse operador mantém pressão seletiva sem depender de probabilidades proporcionais à fitness.


In [7]:
def tournament_selection(population, k=3): 
    candidates = random.sample(population, k)
    return max(candidates, key=lambda x: x["evaluation"])

#### Item 6.3 - Crossover uniforme

O crossover uniforme cria um novo indivíduo misturando genes de dois pais. Para cada hiperparâmetro, o valor é herdado de um dos pais com probabilidade igual.


In [8]:
def uniform_crossover(parent1, parent2, crossover_rate=0.85):
    if random.random() > crossover_rate:
        selected = random.choice([parent1, parent2])
        child = selected.copy()
        return child

    child = {}

    for key in parent1.keys():
        if random.random() < 0.5:
            child[key] = parent1[key]
        else:
            child[key] = parent2[key]

    return child

#### Item 6.4 - Mutação por gene

A mutação altera cada gene com uma probabilidade definida pelo experimento. Esse operador mantém diversidade na população e ajuda a explorar configurações que não apareceriam apenas por crossover.


In [9]:
def rf_mutation(individual, mutation_rate=0.20):
    new = individual.copy()

    if random.random() < mutation_rate:
        new["n_estimators"] = random.choice(N_ESTIMATORS_SPACE)

    if random.random() < mutation_rate:
        new["max_depth"] = random.choice(MAX_DEPTH_SPACE)

    if random.random() < mutation_rate:
        new["min_samples_leaf"] = random.choice(MIN_SAMPLES_LEAF_SPACE)

    if random.random() < mutation_rate:
        new["class_weight"] = random.choice(CLASS_WEIGHT_SPACE)

    return new


#### Item 6.5 - Nova geração com elitismo

A nova geração preserva os melhores indivíduos da geração atual e completa o restante da população com filhos gerados por seleção, crossover e mutação. O elitismo evita perder a melhor solução já encontrada.


In [10]:
def generate_new_generation(
    sorted_by_fitness_current_population,
    population_size,
    crossover_rate=0.85,
    mutation_rate=0.20,
    elite_size=2
):
    new_generation = []    

    
    elite = sorted_by_fitness_current_population[:elite_size]
    new_generation.extend([ind.copy() for ind in elite])

    while len(new_generation) < population_size:
        parent1 = tournament_selection(sorted_by_fitness_current_population)
        parent2 = tournament_selection(sorted_by_fitness_current_population)

        child = uniform_crossover(
            parent1,
            parent2,
            crossover_rate=crossover_rate
        )

        child = rf_mutation(
            child,
            mutation_rate=mutation_rate
        )   
        new_generation.append(child)

    return new_generation

#### Item 6.6 - Execução do Algoritmo Genético com monitoramento

A função principal executa as gerações do Algoritmo Genético. Em cada geração, a população é avaliada, o melhor indivíduo é registrado e o histórico armazena informações para tracking de desempenho: melhor fitness, fitness média, desvio padrão, pior fitness, tempo da geração e configuração do experimento.

Esses registros funcionam como logging estruturado do treinamento e permitem comparar custo computacional e evolução da população entre diferentes configurações do AG.


In [11]:
def extract_fitness_values(population):
    return [individual["evaluation"] for individual in population]


def execute_ga(
    generation_count,
    population_size,
    crossover_rate,
    mutation_rate,
    elite_size,
    experiment_name="Experimento",
):
    current_generation = None
    history_data = []
    best_individuals = []
    experiment_started_at = time.perf_counter()

    for i in range(0, generation_count):
        generation_started_at = time.perf_counter()
        current_generation = generate_new_generation(
            sorted_by_fitness_current_population,
            population_size=population_size,
            crossover_rate=crossover_rate,
            mutation_rate=mutation_rate,
            elite_size=elite_size,
        ) if current_generation is not None else create_hot_start_individuals(population_size)

        sorted_by_fitness_current_population = evaluate_population(current_generation)
        best_evaluation = sorted_by_fitness_current_population[0]
        fitness_values = extract_fitness_values(sorted_by_fitness_current_population)
        generation_elapsed_seconds = time.perf_counter() - generation_started_at

        history_data.append({
            "Experimento": experiment_name,
            "Geração": i + 1,
            "População": population_size,
            "Taxa crossover": crossover_rate,
            "Taxa mutação": mutation_rate,
            "Elitismo": elite_size,
            "Melhor fitness": round(float(max(fitness_values)), 6),
            "Fitness média": round(float(np.mean(fitness_values)), 6),
            "Fitness desvio": round(float(np.std(fitness_values)), 6),
            "Pior fitness": round(float(min(fitness_values)), 6),
            "Tempo geração (s)": round(generation_elapsed_seconds, 3),
            "Melhor Indivíduo": best_evaluation.copy(),
        })
        best_individuals.append(best_evaluation.copy())

    top_individual = sorted(
        best_individuals,
        key=lambda item: item["evaluation"],
        reverse=True,
    )[0]

    experiment_elapsed_seconds = time.perf_counter() - experiment_started_at
    experiment_log = {
        "Experimento": experiment_name,
        "Gerações": generation_count,
        "População": population_size,
        "Taxa crossover": crossover_rate,
        "Taxa mutação": mutation_rate,
        "Elitismo": elite_size,
        "Melhor fitness": round(float(top_individual["evaluation"]), 6),
        "Tempo total (s)": round(experiment_elapsed_seconds, 3),
        "Tempo médio por geração (s)": round(experiment_elapsed_seconds / generation_count, 3),
        "Melhor Indivíduo": top_individual.copy(),
    }

    return pd.DataFrame(history_data), top_individual, experiment_log


#### Item 7 - Execução dos três experimentos e logs de desempenho

O trabalho exige pelo menos três experimentos com configurações diferentes do Algoritmo Genético. As execuções abaixo variam tamanho da população, taxa de mutação e número de gerações, mantendo o mesmo crossover e elitismo.

Além do melhor indivíduo, cada experimento gera logs estruturados para tracking de desempenho: fitness por geração, estatísticas da população e tempo de execução.


In [12]:
experiment_1_history, experiment_1_best, experiment_1_log = execute_ga(
    experiment_name="Experimento 1",
    generation_count=20,
    population_size=20,
    crossover_rate=0.85,
    mutation_rate=0.05,
    elite_size=2,
)

experiment_2_history, experiment_2_best, experiment_2_log = execute_ga(
    experiment_name="Experimento 2",
    generation_count=20,
    population_size=40,
    crossover_rate=0.85,
    mutation_rate=0.10,
    elite_size=2,
)

experiment_3_history, experiment_3_best, experiment_3_log = execute_ga(
    experiment_name="Experimento 3",
    generation_count=30,
    population_size=40,
    crossover_rate=0.85,
    mutation_rate=0.20,
    elite_size=2,
)

ga_experiment_summary = pd.DataFrame([
    experiment_1_log,
    experiment_2_log,
    experiment_3_log,
])

display(ga_experiment_summary)


,Experimento,Gerações,População,Taxa crossover,Taxa mutação,Elitismo,Melhor fitness,Tempo total (s),Tempo médio por geração (s),Melhor Indivíduo
0,Experimento 1,20,20,0.85,0.05,2,0.220586,829.590,41.480,"{'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 20, 'class_weight': 'balanced', 'evaluation': 0.22058649380026235}"
1,Experimento 2,20,40,0.85,0.10,2,0.222286,918.563,45.928,"{'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 10, 'class_weight': 'balanced', 'evaluation': 0.22228605282451624}"
2,Experimento 3,30,40,0.85,0.20,2,0.222286,1886.246,62.875,"{'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 10, 'class_weight': 'balanced', 'evaluation': 0.22228605282451624}"


#### Item 7.1 - Histórico e tracking do Experimento 1

Este histórico mostra o melhor indivíduo encontrado em cada geração do primeiro experimento, junto com métricas de monitoramento da população e tempo por geração.


In [13]:
display(experiment_1_history)

,Experimento,Geração,População,Taxa crossover,Taxa mutação,Elitismo,Melhor fitness,Fitness média,Fitness desvio,Pior fitness,Tempo geração (s),Melhor Indivíduo
0,Experimento 1,1,20,0.85,0.05,2,0.199171,0.162273,0.068296,0.000000,44.984,"{'n_estimators': 200, 'max_depth': 12, 'min_samples_leaf': 10, 'class_weight': 'balanced_subsample', 'evaluation': 0.1991708585438869}"
1,Experimento 1,2,20,0.85,0.05,2,0.213567,0.194640,0.007743,0.188297,50.705,"{'n_estimators': 200, 'max_depth': 8, 'min_samples_leaf': 5, 'class_weight': 'balanced_subsample', 'evaluation': 0.2135665860255304}"
2,Experimento 1,3,20,0.85,0.05,2,0.214970,0.190501,0.042908,0.007461,48.498,"{'n_estimators': 200, 'max_depth': 8, 'min_samples_leaf': 10, 'class_weight': 'balanced_subsample', 'evaluation': 0.2149701609090783}"
3,Experimento 1,4,20,0.85,0.05,2,0.214970,0.193413,0.045082,0.000000,43.676,"{'n_estimators': 200, 'max_depth': 8, 'min_samples_leaf': 10, 'class_weight': 'balanced_subsample', 'evaluation': 0.2149701609090783}"
4,Experimento 1,5,20,0.85,0.05,2,0.217529,0.210021,0.007469,0.191311,45.929,"{'n_estimators': 200, 'max_depth': 8, 'min_samples_leaf': 20, 'class_weight': 'balanced_subsample', 'evaluation': 0.21752906639809066}"
5,Experimento 1,6,20,0.85,0.05,2,0.217529,0.204440,0.046925,0.000000,46.989,"{'n_estimators': 200, 'max_depth': 8, 'min_samples_leaf': 20, 'class_weight': 'balanced_subsample', 'evaluation': 0.21752906639809066}"
6,Experimento 1,7,20,0.85,0.05,2,0.217561,0.216635,0.001222,0.214970,47.574,"{'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 5, 'class_weight': 'balanced_subsample', 'evaluation': 0.2175606482076991}"
7,Experimento 1,8,20,0.85,0.05,2,0.217561,0.216535,0.003800,0.200153,43.711,"{'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 5, 'class_weight': 'balanced_subsample', 'evaluation': 0.2175606482076991}"
8,Experimento 1,9,20,0.85,0.05,2,0.217561,0.216749,0.001591,0.213567,37.166,"{'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 5, 'class_weight': 'balanced_subsample', 'evaluation': 0.2175606482076991}"
9,Experimento 1,10,20,0.85,0.05,2,0.219422,0.217837,0.000666,0.217529,26.985,"{'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 20, 'class_weight': 'balanced_subsample', 'evaluation': 0.21942195155104188}"


#### Item 7.2 - Histórico e tracking do Experimento 2

Este histórico mostra o melhor indivíduo encontrado em cada geração do segundo experimento, junto com métricas de monitoramento da população e tempo por geração.


In [14]:
display(experiment_2_history)

,Experimento,Geração,População,Taxa crossover,Taxa mutação,Elitismo,Melhor fitness,Fitness média,Fitness desvio,Pior fitness,Tempo geração (s),Melhor Indivíduo
0,Experimento 2,1,40,0.85,0.1,2,0.219370,0.186621,0.033456,0.000000,92.852,"{'n_estimators': 200, 'max_depth': 8, 'min_samples_leaf': 10, 'class_weight': 'balanced', 'evaluation': 0.21937026530439363}"
1,Experimento 2,2,40,0.85,0.1,2,0.219370,0.187535,0.042447,0.000000,74.432,"{'n_estimators': 200, 'max_depth': 8, 'min_samples_leaf': 10, 'class_weight': 'balanced', 'evaluation': 0.21937026530439363}"
2,Experimento 2,3,40,0.85,0.1,2,0.221989,0.210192,0.010249,0.188066,79.935,"{'n_estimators': 200, 'max_depth': 8, 'min_samples_leaf': 20, 'class_weight': 'balanced', 'evaluation': 0.2219887806252813}"
3,Experimento 2,4,40,0.85,0.1,2,0.222286,0.206108,0.047939,0.000000,68.718,"{'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 10, 'class_weight': 'balanced', 'evaluation': 0.22228605282451624}"
4,Experimento 2,5,40,0.85,0.1,2,0.222286,0.219174,0.005578,0.191326,56.980,"{'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 10, 'class_weight': 'balanced', 'evaluation': 0.22228605282451624}"
5,Experimento 2,6,40,0.85,0.1,2,0.222286,0.217339,0.011017,0.180753,48.023,"{'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 10, 'class_weight': 'balanced', 'evaluation': 0.22228605282451624}"
6,Experimento 2,7,40,0.85,0.1,2,0.222286,0.217828,0.008855,0.182960,40.353,"{'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 10, 'class_weight': 'balanced', 'evaluation': 0.22228605282451624}"
7,Experimento 2,8,40,0.85,0.1,2,0.222286,0.212074,0.035486,0.000000,40.442,"{'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 10, 'class_weight': 'balanced', 'evaluation': 0.22228605282451624}"
8,Experimento 2,9,40,0.85,0.1,2,0.222286,0.197818,0.066322,0.000000,33.931,"{'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 10, 'class_weight': 'balanced', 'evaluation': 0.22228605282451624}"
9,Experimento 2,10,40,0.85,0.1,2,0.222286,0.217192,0.011200,0.182960,35.493,"{'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 10, 'class_weight': 'balanced', 'evaluation': 0.22228605282451624}"


#### Item 7.3 - Histórico e tracking do Experimento 3

Este histórico mostra o melhor indivíduo encontrado em cada geração do terceiro experimento, junto com métricas de monitoramento da população e tempo por geração.


In [15]:
display(experiment_3_history)

,Experimento,Geração,População,Taxa crossover,Taxa mutação,Elitismo,Melhor fitness,Fitness média,Fitness desvio,Pior fitness,Tempo geração (s),Melhor Indivíduo
0,Experimento 3,1,40,0.85,0.2,2,0.219552,0.178423,0.041338,0.00000,99.198,"{'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 10, 'class_weight': 'balanced_subsample', 'evaluation': 0.21955187588257885}"
1,Experimento 3,2,40,0.85,0.2,2,0.219858,0.178805,0.060719,0.00000,105.325,"{'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 20, 'class_weight': 'balanced_subsample', 'evaluation': 0.21985762975512982}"
2,Experimento 3,3,40,0.85,0.2,2,0.219858,0.190920,0.044477,0.00000,116.506,"{'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 20, 'class_weight': 'balanced_subsample', 'evaluation': 0.21985762975512982}"
3,Experimento 3,4,40,0.85,0.2,2,0.219858,0.198524,0.048281,0.00000,113.950,"{'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 20, 'class_weight': 'balanced_subsample', 'evaluation': 0.21985762975512982}"
4,Experimento 3,5,40,0.85,0.2,2,0.220586,0.204419,0.036596,0.00000,112.627,"{'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 20, 'class_weight': 'balanced', 'evaluation': 0.22058649380026235}"
5,Experimento 3,6,40,0.85,0.2,2,0.220586,0.205143,0.047800,0.00000,110.162,"{'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 20, 'class_weight': 'balanced', 'evaluation': 0.22058649380026235}"
6,Experimento 3,7,40,0.85,0.2,2,0.221989,0.201330,0.049271,0.00000,98.483,"{'n_estimators': 200, 'max_depth': 8, 'min_samples_leaf': 20, 'class_weight': 'balanced', 'evaluation': 0.2219887806252813}"
7,Experimento 3,8,40,0.85,0.2,2,0.221989,0.188918,0.071949,0.00000,86.060,"{'n_estimators': 200, 'max_depth': 8, 'min_samples_leaf': 20, 'class_weight': 'balanced', 'evaluation': 0.2219887806252813}"
8,Experimento 3,9,40,0.85,0.2,2,0.221989,0.203952,0.047622,0.00000,76.064,"{'n_estimators': 200, 'max_depth': 8, 'min_samples_leaf': 20, 'class_weight': 'balanced', 'evaluation': 0.2219887806252813}"
9,Experimento 3,10,40,0.85,0.2,2,0.222286,0.212252,0.034820,0.00000,68.807,"{'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 10, 'class_weight': 'balanced', 'evaluation': 0.22228605282451624}"


#### Item 7.4 - Validação cruzada do melhor indivíduo do Experimento 1

A célula abaixo reexibe as métricas médias de validação cruzada para o melhor indivíduo encontrado no primeiro experimento.


In [16]:
execute_random_forest(
    x=X_train_sample,
    y=y_train_sample,
    n_estimators=experiment_1_best["n_estimators"],
    max_depth=experiment_1_best["max_depth"],
    min_samples_leaf=experiment_1_best["min_samples_leaf"],
    class_weight=experiment_1_best["class_weight"],
    show=True,
)


########### RandomForestClassifier ##############
n_estimators: 300
max_depth: 8
min_samples_leaf: 20
class_weight: balanced
accuracy: 0.766625
recall: 0.4241721132897604
f1: 0.13333551401904892
precision: 0.07912588932662484
roc_auc: 0.6384960206516376
balanced_accuracy: 0.602987564019901
#################################################


ClassificationMetrics(accuracy=0.766625, recall=0.4241721132897604, f1=0.13333551401904892, precision=0.07912588932662484, roc_auc=0.6384960206516376, balanced_accuracy=0.602987564019901)

#### Item 7.5 - Validação cruzada do melhor indivíduo do Experimento 2

A célula abaixo reexibe as métricas médias de validação cruzada para o melhor indivíduo encontrado no segundo experimento.


In [17]:
execute_random_forest(
    x=X_train_sample,
    y=y_train_sample,
    n_estimators=experiment_2_best["n_estimators"],
    max_depth=experiment_2_best["max_depth"],
    min_samples_leaf=experiment_2_best["min_samples_leaf"],
    class_weight=experiment_2_best["class_weight"],
    show=True,
)


########### RandomForestClassifier ##############
n_estimators: 100
max_depth: 8
min_samples_leaf: 10
class_weight: balanced
accuracy: 0.7707499999999999
recall: 0.4241830065359477
f1: 0.1357587869481885
precision: 0.08083768126305581
roc_auc: 0.6318632347690097
balanced_accuracy: 0.6051469618055773
#################################################


ClassificationMetrics(accuracy=0.7707499999999999, recall=0.4241830065359477, f1=0.1357587869481885, precision=0.08083768126305581, roc_auc=0.6318632347690097, balanced_accuracy=0.6051469618055773)

#### Item 7.6 - Validação cruzada do melhor indivíduo do Experimento 3

A célula abaixo reexibe as métricas médias de validação cruzada para o melhor indivíduo encontrado no terceiro experimento.


In [18]:
execute_random_forest(
    x=X_train_sample,
    y=y_train_sample,
    n_estimators=experiment_3_best["n_estimators"],
    max_depth=experiment_3_best["max_depth"],
    min_samples_leaf=experiment_3_best["min_samples_leaf"],
    class_weight=experiment_3_best["class_weight"],
    show=True,
)


########### RandomForestClassifier ##############
n_estimators: 100
max_depth: 8
min_samples_leaf: 10
class_weight: balanced
accuracy: 0.7707499999999999
recall: 0.4241830065359477
f1: 0.1357587869481885
precision: 0.08083768126305581
roc_auc: 0.6318632347690097
balanced_accuracy: 0.6051469618055773
#################################################


ClassificationMetrics(accuracy=0.7707499999999999, recall=0.4241830065359477, f1=0.1357587869481885, precision=0.08083768126305581, roc_auc=0.6318632347690097, balanced_accuracy=0.6051469618055773)

#### Item 8 - Avaliação final da Random Forest otimizada

Depois dos três experimentos, o melhor indivíduo é escolhido pela maior fitness média. O modelo final é treinado uma única vez com a amostra de treino e avaliado na amostra estratificada de teste.

Nesta etapa não há validação cruzada: o teste é usado apenas para medir o desempenho final do modelo otimizado pelo Algoritmo Genético.


In [19]:
best_individual = max([experiment_1_best, experiment_2_best, experiment_3_best], key=lambda x: x["evaluation"])

final_model = RandomForestClassifier(
    n_estimators=best_individual["n_estimators"],
    max_depth=best_individual["max_depth"],
    min_samples_leaf=best_individual["min_samples_leaf"],
    class_weight=best_individual["class_weight"],
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

final_model.fit(X_train_sample, y_train_sample)
y_pred = final_model.predict(X_test_sample)
y_score = final_model.predict_proba(X_test_sample)[:, 1]
current_execution = calculate_metrics(y_test_sample, y_pred, y_score)
display(current_execution)


ClassificationMetrics(accuracy=0.760875, recall=0.4247787610619469, f1=0.1308496138119037, precision=0.07733619763694952, roc_auc=0.6288853746844052, balanced_accuracy=0.6002630262691278)

#### Item 9 - Comparação com o baseline de Random Forest da Parte 5

A comparação abaixo usa como referência a Random Forest tunada com `GridSearchCV` na Parte 5. Como a base é desbalanceada, a leitura principal deve priorizar F1, recall, precision e balanced accuracy, e não apenas accuracy.

O modelo otimizado por Algoritmo Genético é avaliado com o melhor indivíduo encontrado nos três experimentos anteriores, usando uma amostra estratificada de teste com 8.000 registros, mesma escala usada no baseline da Parte 5.


In [20]:
baseline_part_5_metrics = {
    "modelo": "Random Forest baseline Parte 5 (GridSearchCV)",
    "accuracy": 0.7971,
    "balanced_accuracy": 0.5924,
    "precision": 0.0815,
    "recall": 0.3687,
    "f1": 0.1335,
    "roc_auc": 0.6248,
}

optimized_ga_metrics = {
    "modelo": "Random Forest otimizada com Algoritmo Genético",
    "accuracy": round(current_execution.accuracy, 4),
    "balanced_accuracy": round(current_execution.balanced_accuracy, 4),
    "precision": round(current_execution.precision, 4),
    "recall": round(current_execution.recall, 4),
    "f1": round(current_execution.f1, 4),
    "roc_auc": round(current_execution.roc_auc, 4),
}

metrics_comparison = pd.DataFrame([
    baseline_part_5_metrics,
    optimized_ga_metrics,
])

metrics_delta = {
    "modelo": "Diferença AG - baseline",
    "accuracy": round(optimized_ga_metrics["accuracy"] - baseline_part_5_metrics["accuracy"], 4),
    "balanced_accuracy": round(optimized_ga_metrics["balanced_accuracy"] - baseline_part_5_metrics["balanced_accuracy"], 4),
    "precision": round(optimized_ga_metrics["precision"] - baseline_part_5_metrics["precision"], 4),
    "recall": round(optimized_ga_metrics["recall"] - baseline_part_5_metrics["recall"], 4),
    "f1": round(optimized_ga_metrics["f1"] - baseline_part_5_metrics["f1"], 4),
    "roc_auc": round(optimized_ga_metrics["roc_auc"] - baseline_part_5_metrics["roc_auc"], 4),
}

metrics_comparison = pd.concat(
    [metrics_comparison, pd.DataFrame([metrics_delta])],
    ignore_index=True,
)

metrics_comparison


,modelo,accuracy,balanced_accuracy,precision,recall,f1,roc_auc
0,Random Forest baseline Parte 5 (GridSearchCV),0.7971,0.5924,0.0815,0.3687,0.1335,0.6248
1,Random Forest otimizada com Algoritmo Genético,0.7609,0.6003,0.0773,0.4248,0.1308,0.6289
2,Diferença AG - baseline,-0.0362,0.0079,-0.0042,0.0561,-0.0027,0.0041


#### Item 9.1 - Relatório final da Random Forest otimizada

O relatório de classificação e a matriz de confusão detalham o comportamento do modelo final na amostra de teste. Esses resultados ajudam a interpretar o trade-off entre recuperar mais casos positivos e manter precisão aceitável.


In [21]:
final_classification_report = pd.DataFrame(
    classification_report(
        y_test_sample,
        y_pred,
        output_dict=True,
        zero_division=0,
    )
).T

final_confusion_matrix = pd.DataFrame(
    confusion_matrix(y_test_sample, y_pred),
    index=["real_0", "real_1"],
    columns=["pred_0", "pred_1"],
)

display(final_classification_report)
display(final_confusion_matrix)


,precision,recall,f1-score,support
0,0.968231,0.775747,0.861367,7661.000000
1,0.077336,0.424779,0.130850,339.000000
accuracy,0.760875,0.760875,0.760875,0.760875
macro avg,0.522783,0.600263,0.496108,8000.000000
weighted avg,0.930479,0.760875,0.830411,8000.000000


,pred_0,pred_1
real_0,5943,1718
real_1,195,144


#### Item 9.2 - Conclusão da otimização com Algoritmo Genético

A conclusão abaixo sintetiza a melhor configuração encontrada pelo Algoritmo Genético, compara o resultado com o baseline da Parte 5 e registra a interpretação das métricas mais importantes para a classe positiva.


In [22]:
f1_delta = metrics_delta["f1"]
recall_delta = metrics_delta["recall"]
precision_delta = metrics_delta["precision"]
balanced_accuracy_delta = metrics_delta["balanced_accuracy"]

conclusion_rows = [
    {
        "aspecto": "melhor_individuo_ag",
        "conclusao": f"n_estimators={best_individual['n_estimators']}, max_depth={best_individual['max_depth']}, min_samples_leaf={best_individual['min_samples_leaf']}, class_weight={best_individual['class_weight']}, fitness={best_individual['evaluation']:.4f}.",
    },
    {
        "aspecto": "comparacao_f1",
        "conclusao": f"O F1 da Random Forest com AG foi {optimized_ga_metrics['f1']:.4f}, contra {baseline_part_5_metrics['f1']:.4f} no baseline da Parte 5. Diferença: {f1_delta:+.4f}.",
    },
    {
        "aspecto": "comparacao_recall",
        "conclusao": f"O recall da Random Forest com AG foi {optimized_ga_metrics['recall']:.4f}, contra {baseline_part_5_metrics['recall']:.4f} no baseline. Diferença: {recall_delta:+.4f}.",
    },
    {
        "aspecto": "comparacao_precision",
        "conclusao": f"A precision da Random Forest com AG foi {optimized_ga_metrics['precision']:.4f}, contra {baseline_part_5_metrics['precision']:.4f} no baseline. Diferença: {precision_delta:+.4f}.",
    },
    {
        "aspecto": "balanced_accuracy",
        "conclusao": f"A balanced accuracy da Random Forest com AG foi {optimized_ga_metrics['balanced_accuracy']:.4f}, contra {baseline_part_5_metrics['balanced_accuracy']:.4f} no baseline. Diferença: {balanced_accuracy_delta:+.4f}.",
    },
    {
        "aspecto": "interpretacao",
        "conclusao": "Como se trata de um problema de saúde, a redução de falsos negativos é importante. O resultado deve ser lido como trade-off entre recall, precision, F1 e balanced accuracy, sem usar accuracy isolada como critério principal.",
    },
    {
        "aspecto": "decisao_final",
        "conclusao": "A Random Forest otimizada por Algoritmo Genético deve ser comparada ao baseline da Parte 5 e aos demais modelos otimizados. A decisão final depende do equilíbrio entre sensibilidade para a classe positiva e volume de falsos positivos.",
    },
]

optimization_conclusion = pd.DataFrame(conclusion_rows)
optimization_conclusion


,aspecto,conclusao
0,melhor_individuo_ag,"n_estimators=100, max_depth=8, min_samples_leaf=10, class_weight=balanced, fitness=0.2223."
1,comparacao_f1,"O F1 da Random Forest com AG foi 0.1308, contra 0.1335 no baseline da Parte 5. Diferença: -0.0027."
2,comparacao_recall,"O recall da Random Forest com AG foi 0.4248, contra 0.3687 no baseline. Diferença: +0.0561."
3,comparacao_precision,"A precision da Random Forest com AG foi 0.0773, contra 0.0815 no baseline. Diferença: -0.0042."
4,balanced_accuracy,"A balanced accuracy da Random Forest com AG foi 0.6003, contra 0.5924 no baseline. Diferença: +0.0079."
5,interpretacao,"Como se trata de um problema de saúde, a redução de falsos negativos é importante. O resultado deve ser lido como trade-off entre recall, precision, F1 e balanced accuracy, sem usar accuracy isolada como critério principal."
6,decisao_final,A Random Forest otimizada por Algoritmo Genético deve ser comparada ao baseline da Parte 5 e aos demais modelos otimizados. A decisão final depende do equilíbrio entre sensibilidade para a classe positiva e volume de falsos positivos.


---

## Resultado da Parte 9

Este notebook implementa um Algoritmo Genético para otimizar hiperparâmetros da Random Forest (`RandomForestClassifier`) usando as bases encoded finais da Parte 2.

A avaliação dos indivíduos ocorre por validação cruzada estratificada com 5 folds, usando uma fitness voltada à classe positiva:

```text
fitness = 0.7 * F1 + 0.3 * recall
```

Foram definidos três experimentos com diferentes configurações de população, mutação e gerações. Ao final, o melhor indivíduo entre os experimentos é usado para treinar um modelo final e avaliar seu desempenho na amostra de teste.

A seção final compara a Random Forest otimizada por Algoritmo Genético com o baseline de Random Forest da Parte 5, usando a mesma escala de amostra de teste. A interpretação prioriza métricas adequadas para desbalanceamento, especialmente recall, F1, precision e balanced accuracy.

A conclusão operacional deve ser lida a partir da tabela `optimization_conclusion`. Se o recall do modelo com AG ficar acima do baseline da Parte 5, isso indica maior sensibilidade para capturar positivos. Caso isso venha acompanhado de queda em precision, F1 ou balanced accuracy, o resultado deve ser apresentado como um trade-off esperado em triagem de saúde: maior sensibilidade para capturar positivos, com mais falsos positivos e necessidade de confirmação diagnóstica posterior.

O monitoramento do AG é registrado em dois níveis: `experiment_1_history`, `experiment_2_history` e `experiment_3_history` armazenam a evolução por geração; `ga_experiment_summary` consolida a configuração, o melhor fitness e o tempo total de cada experimento. Esses logs permitem rastrear desempenho, convergência e custo computacional entre as configurações testadas.

